# 1. Stiffness, Eigenvalues

In [ ]:
from main import *

#####Given Functions#####
def forward_euler(f, u_start, t_end, dt):
    """March u' = f(u, t) from t = 0 to t_end with forward Euler (Lecture 10).

    Returns
    -------
    u : ndarray, shape (n + 1, len(u_start)); row k is the state at t_k
    times : ndarray, the n + 1 times t_k
    """
    n = int(round(t_end / dt))
    u = np.empty((n + 1, np.size(u_start)))
    u[0] = u_start
    for k in range(n):
        u[k + 1] = u[k] + dt * f(u[k], k * dt)
    return u, np.arange(n + 1) * dt

# Object

In [ ]:
Ar = 1.8e13 / np.sqrt(1000.0)
Ea = 17614.0 * 8.314e3
tube = chamber_solver(5, 3e6, 0.3, 1, Ar, Ea)

# Forward Euler & Stifness Visualization

In [ ]:
E, Rs, dx_lim = tube.stability_along_solution()

s0, R, gamma, h1, h2, h3 = tube.initial_conditions()

f_fn = lambda *u: tube.rhs(u, R, gamma, h1, h2, h3)

dx_small, dx_large, dx_ref = 0.99*dx_lim, 1.01*dx_lim, 0.03*dx_lim
L = 1e-4

rhs_tube = lambda u, x: np.array(f_fn(*u), float).ravel()

fig, (ax, axY2, axT, axR, axV) = plt.subplots(5, 1, figsize=(8, 17), sharex=True)
for dx, ls, lab in [(dx_small, ':', 'below limit'), (dx_large, '--', 'above limit'), (dx_ref, '-', 'reference')]:
    U, x = forward_euler(rhs_tube, s0, L, dx)
    lbl = f"dx = {dx:.1e} m ({lab})"
    ax.plot(x*1e6,   U[:, 3], ls, label=lbl)
    axY2.plot(x*1e6, U[:, 4], ls, label=lbl)
    axT.plot(x*1e6,  U[:, 1], ls, label=lbl)
    axR.plot(x*1e6,  U[:, 0], ls, label=lbl)
    axV.plot(x*1e6,  U[:, 2], ls, label=lbl)
    if lab == 'reference':
        x_ref, U_ref = x, U

xr  = x_ref * 1e6
Y1r = U_ref[:, 3]
x_burn = xr[np.argmax(Y1r < 0.5)]
x_end  = xr[np.argmax(Y1r < 1e-3)]
pad = max(x_end - x_burn, 0.05 * x_burn)

ax.set_ylabel("Y1")
ax.set_ylim(-0.1, 1.1)
ax.legend(); ax.grid(alpha=0.3)

axY2.set_ylabel("Y2")
axY2.set_ylim(-0.1, 1.1)
axY2.grid(alpha=0.3)

axT.set_ylabel("T (K)")
axT.set_ylim(0, 6000)
axT.grid(alpha=0.3)

axR.set_ylabel("ρ (kg/m³)")
axR.grid(alpha=0.3)

axV.set_ylabel("V (m/s)")
axV.set_xlabel("x (µm)")
axV.grid(alpha=0.3)

ax.set_xlim(max(x_burn - 3*pad, 0), x_end + 5*pad)
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
import numpy.linalg as la
import scipy as sc
import scipy.optimize
import cantera as ct
import matplotlib.pyplot as plt

from main import chamber_solver

Ar = 1.8e13 / np.sqrt(1000.0)
Ea = 17614.0 * 8.314e3
solver = chamber_solver(5, 3e6, 0.3, 1, Ar, Ea)

state0, R, gamma, h1, h2, h3 = solver.initial_conditions()
rho0, T0, V0, Y10, Y20, Y30 = state0

gas = ct.Solution("h2o2.yaml")
i1 = gas.species_index("O2")
i2 = gas.species_index("H2")
i3 = gas.species_index("H2O")

def p_of(u):
    rho, T, V, Y1, Y2, Y3 = u
    Mw = 1.0 / (Y1/solver.MW_O2 + Y2/solver.MW_H2 + Y3/solver.MW_H2O)
    return rho * (ct.gas_constant / Mw) * T

def h_of(u):
    rho, T, V, Y1, Y2, Y3 = u
    gas.TPY = T, solver.Pt, {"O2": Y1, "H2": Y2, "H2O": Y3}
    h = gas.partial_molar_enthalpies / gas.molecular_weights
    return Y1*h[i1] + Y2*h[i2] + Y3*h[i3] + 0.5*V*V

def partial_react_state(Y1, Y2, guess):
    m0   = rho0 * V0
    mom0 = p_of(state0) + rho0 * V0**2
    h0   = h_of(state0)
    Y3   = 1.0 - Y1 - Y2
    def cons(z):
        ro, Tt, Vv = z
        u = np.array([ro, Tt, Vv, Y1, Y2, Y3])
        return [ro*Vv - m0,
                p_of(u) + ro*Vv**2 - mom0,
                h_of(u) - h0]
    ro, Tt, Vv = sc.optimize.fsolve(cons, guess)
    return np.array([ro, Tt, Vv, Y1, Y2, Y3])

s_inlet   = state0.copy()
s_partial = partial_react_state(0.5, 0.5, [2*rho0, 2*T0, 2*V0])
s_near    = partial_react_state(0.1, 0.9, [2*rho0, 2*T0, 2*V0])
s_steady  = partial_react_state(0.0, 0.0, [2*rho0, 2*T0, 2*V0])

names  = ["inlet", "partial", "near steady", "steady"]
states = [s_inlet, s_partial, s_near, s_steady]

def analyze(u):
    J = solver.jacobian(u, R, gamma, h1, h2, h3)
    E = np.sort(np.abs(la.eigvals(J)))[::-1]
    nz = E[E > 1e-8 * E[0]]
    return E, nz[0] / nz[-1], 2 / nz[0]

print(f"{'case':<12}{'rho':>9}{'T':>9}{'V':>9}{'Y1':>7}{'Y2':>7}{'Y3':>7}"
      f"{'|l1|':>12}{'|l2|':>12}{'|l3|':>12}{'|l4|':>12}{'|l5|':>12}{'|l6|':>12}{'R_s':>12}{'dx_max(m)':>13}")
print("-" * 168)
table = {}
for n, u in zip(names, states):
    E, Rs, dx = analyze(u)
    table[n] = (u, E, Rs, dx)
    Ep = np.pad(E, (0, max(0, 6-len(E))), constant_values=np.nan)
    print(f"{n:<12}{u[0]:9.3f}{u[1]:9.1f}{u[2]:9.2f}{u[3]:7.3f}{u[4]:7.3f}{u[5]:7.3f}"
          + "".join(f"{e:12.3e}" for e in Ep[:6])
          + f"{Rs:12.3e}{dx:13.3e}")

case = "steady"
u_case, E_case, Rs_case, dx_lim = table[case]
print(f"\ncase = {case}, dx_lim = {dx_lim:.3e} m")

dx_small, dx_large, dx_ref = 0.99*dx_lim, 1.01*dx_lim, 0.03*dx_lim
L = 1e-4

rhs_tube = lambda u: solver.rhs(u, R, gamma, h1, h2, h3)

def forward_euler(rhs, u0, L, dx):
    n = max(int(round(L/dx)), 1)
    x = np.linspace(0.0, L, n+1)
    U = np.empty((n+1, len(u0)))
    U[0] = u0
    for k in range(n):
        U[k+1] = U[k] + dx * np.asarray(rhs(U[k]), float).ravel()
    return U, x

fig, (ax, axY2, axY3, axT, axR, axV) = plt.subplots(6, 1, figsize=(8, 20), sharex=True)
for dx, ls, lab in [(dx_small, ':', 'below limit'), (dx_large, '--', 'above limit'), (dx_ref, '-', 'reference')]:
    U, x = forward_euler(rhs_tube, u_case, L, dx)
    lbl = f"dx = {dx:.1e} m ({lab})"
    ax.plot(x*1e6,   U[:, 3], ls, label=lbl)
    axY2.plot(x*1e6, U[:, 4], ls, label=lbl)
    axY3.plot(x*1e6, U[:, 5], ls, label=lbl)
    axT.plot(x*1e6,  U[:, 1], ls, label=lbl)
    axR.plot(x*1e6,  U[:, 0], ls, label=lbl)
    axV.plot(x*1e6,  U[:, 2], ls, label=lbl)
    if lab == 'reference':
        x_ref, U_ref = x, U

xr  = x_ref * 1e6
ax.set_ylabel("Y1"); ax.set_ylim(-0.1, 1.1); ax.legend(); ax.grid(alpha=0.3)
axY2.set_ylabel("Y2"); axY2.set_ylim(-0.1, 1.1); axY2.grid(alpha=0.3)
axY3.set_ylabel("Y3"); axY3.set_ylim(-0.1, 1.1); axY3.grid(alpha=0.3)
axT.set_ylabel("T (K)"); axT.set_ylim(0, 6000); axT.grid(alpha=0.3)
axR.set_ylabel("rho (kg/m^3)"); axR.set_ylim(0, 1.5*max(u_case[0], rho0)); axR.grid(alpha=0.3)
axV.set_ylabel("V (m/s)"); axV.set_xlabel("x (um)")
axV.set_ylim(0, 2*max(u_case[2], V0)); axV.grid(alpha=0.3)
plt.tight_layout(); plt.show()